In [1]:
from pathlib import Path

import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from xgboost import XGBClassifier

RAW_DATA_PATH = Path("data/creditcard_1.csv")

TARGET_COLUMN = "Class"

RANDOM_STATE = 42

In [ ]:

df = pd.read_csv(RAW_DATA_PATH)
df = df.sort_values("Time").reset_index(drop = True)

target_column = "Class"

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df.drop(columns=[target_column])
y_train = train_df[target_column]

X_test = test_df.drop(columns=[target_column])
y_test = test_df[target_column]

11. Understand scale_pos_weight

This is the XGBoost-specific imbalance mechanism.

For binary classification, a common starting point is:

$$ scale\_pos\_weight \approx \frac{\text{number of negative samples}} {\text{number of positive samples}} $$

In your problem:

In [3]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight_baseline = negative_count / positive_count

print(
    f"Baseline scale_pos_weight: "
    f"{scale_pos_weight_baseline:.2f}"
)

Baseline scale_pos_weight: 545.39


In [4]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)

In [5]:
param_distributions = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1, 0.2],
    "n_estimators": [100, 200, 300],
    "scale_pos_weight": [
        0.5 * scale_pos_weight_baseline,
        scale_pos_weight_baseline,
        2.0 * scale_pos_weight_baseline,
    ],
}

In [6]:
cv = TimeSeriesSplit(n_splits=5)
for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(X_train),
    start=1,
):
    y_validation = y_train.iloc[validation_indices]

    print(
        f"Fold {fold_number}: "
        f"train={len(train_indices)}, "
        f"validation={len(validation_indices)}, "
        f"fraud in validation={y_validation.sum()}"
    )

Fold 1: train=37975, validation=37974, fraud in validation=81
Fold 2: train=75949, validation=37974, fraud in validation=57
Fold 3: train=113923, validation=37974, fraud in validation=86
Fold 4: train=151897, validation=37974, fraud in validation=43
Fold 5: train=189871, validation=37974, fraud in validation=47


In [7]:
search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

In [8]:
search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=Non...
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'learning_rate': [0.03, 0.1, 0.2],
                                        'max_depth': [3, 5, 7],
                                        'n_estimators': [100, 200, 300],
                                        'scale_pos_weight': [272.6954436450839,
                                                             545.3908872901678,
                                                             1090.7817745803357]},
                   random_state=42, return_train_score=True,
                   scoring='average_precision', verbose=1)

### Best Configuration

In [9]:
print("Best parameters:")
print(search.best_params_)

print(
    f"\nBest mean CV Average Precision: "
    f"{search.best_score_:.4f}"
)

Best parameters:
{'scale_pos_weight': 545.3908872901678, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.2}

Best mean CV Average Precision: 0.8111


In [11]:
cv_results = (
    pd.DataFrame(search.cv_results_).reset_index()
    .sort_values("rank_test_score")
)
cv_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_max_depth",
        "param_learning_rate",
        "param_n_estimators",
        "param_scale_pos_weight",
    ]
].head(10)

,mean_test_score,std_test_score,param_max_depth,param_learning_rate,param_n_estimators,param_scale_pos_weight
17,0.811128,0.025318,3,0.2,300,545.390887
14,0.810491,0.025409,5,0.2,200,1090.781775
7,0.810362,0.020729,5,0.2,300,545.390887
12,0.809230,0.020946,5,0.2,200,545.390887
3,0.802634,0.034401,3,0.1,200,545.390887
11,0.802321,0.037077,3,0.1,300,272.695444
13,0.800782,0.032116,3,0.1,300,1090.781775
10,0.799674,0.032572,7,0.1,200,545.390887
0,0.798904,0.035213,3,0.1,200,272.695444
16,0.797312,0.028668,7,0.2,100,545.390887


In [15]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred_proba

array([1.1582948e-05, 1.1980546e-05, 1.2057538e-08, ..., 5.4878610e-06,
       5.5052413e-07, 1.6479133e-04], dtype=float32)

In [17]:
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

test_average_precision = average_precision_score(
    y_test,
    y_pred_proba,
)

print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1: {test_f1:.4f}")
print(
    f"Average Precision: "
    f"{test_average_precision:.4f}"
)

Precision: 0.8235
Recall: 0.7467
F1: 0.7832
Average Precision: 0.7968
